In [0]:
# Live 4 (O) — Orquestração
# Notebook: 21_gold_vendas_kpis_mensais
# Objetivo: KPIs mensais (mesma definição da Live 2).

from pyspark.sql import functions as F

dbutils.widgets.text("tgt_schema", "aulas_ao_vivo.live_20260323")
dbutils.widgets.text("valid_status", "pago")

TGT_SCHEMA = dbutils.widgets.get("tgt_schema")
VALID_STATUS = dbutils.widgets.get("valid_status")

TBL_SILVER_PEDIDOS = f"{TGT_SCHEMA}.silver_vendas_pedidos"
TBL_GOLD = f"{TGT_SCHEMA}.gold_vendas_kpis_mensais"

pedidos = spark.table(TBL_SILVER_PEDIDOS)
base_gold = pedidos.filter(F.col("status") == F.lit(VALID_STATUS))

gold = (
    base_gold
    .groupBy("mes_referencia")
    .agg(
        F.round(F.sum("valor_total_pedido"), 2).alias("receita_total"),
        F.countDistinct("id_pedido").alias("qtd_pedidos_validos"),
        F.sum("quantidade_total").alias("qtd_itens"),
        F.round(F.avg("valor_total_pedido"), 2).alias("ticket_medio_pedido")
    )
    .orderBy("mes_referencia")
)

gold.write.mode("overwrite").format("delta").saveAsTable(TBL_GOLD)

out = spark.table(TBL_GOLD)
out.printSchema()
display(out.orderBy(F.col("mes_referencia").asc()).limit(50))
